<h2>Context</h2> 

- This notebook helps to get the city-wise customers appography category information to build temporary dataset.

## Imports & Connection

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [2]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [6]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)
local_datasets = '/Users/rapido/local-datasets/customer/appography/appography_data_dump/appography-city-datasets/kochi/'
print(local_datasets)

data_dumper_path = '/Users/rapido/local-datasets/customer/appography/appography_data_dump/'

/Users/rapido/analytics/customer/Appography/appography_data_dump
/Users/rapido/local-datasets/customer/appography/appography_data_dump/appography-city-datasets/kochi/


## Parameter & Datasets

In [7]:
start_date = '20240301' 
end_date = '20240331'
city = 'kochi'

### City Base Customer

- City-wise android fare estimate customers taked from the recent month to built the datasets.

- Criteria 

        - Andrioid Customer
        - Minimum 1 FE in recent month

In [8]:
city_fe_customer_query = f"""

        SELECT
            city,
            user_id as user_id
        FROM
            hive.pricing.fare_estimates_enriched
        WHERE 
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND lower(city) = '{city}'
            AND platform = 'android'
        GROUP BY 1,2
"""

df_city_fe_customer = pd.read_sql(city_fe_customer_query, connection)
df_city_fe_customer.to_csv(local_datasets + '{}_city_fe_customer_{}_{}.csv'.format(city,start_date, end_date), index=False)

In [9]:
df_city_fe_customer = pd.read_csv(local_datasets + '{}_city_fe_customer_{}_{}.csv'.format(city,start_date, end_date))
print(df_city_fe_customer.head(3))
df_city_fe_customer.info()

    city                   user_id
0  Kochi  5c7cd474875c98536260755b
1  Kochi  65eac9f1e13df5f3f83c6b16
2  Kochi  62c1bf2717af9a25bb693aa5
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17196 entries, 0 to 17195
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   city     17196 non-null  object
 1   user_id  17196 non-null  object
dtypes: object(2)
memory usage: 268.8+ KB


### Base Customers Recent Appograpgy Data

- Read the local appography data dumper and filter the data by selected city customers

In [10]:
start_time = time.time()

csv_files = [file for file in os.listdir(data_dumper_path) if file.endswith('.csv')]

dfs = []

# Loop through each CSV file, read it into a DataFrame, and append to the list
for file in csv_files:
    file_path = os.path.join(data_dumper_path, file)
    df = pd.read_csv(file_path)
    dfs.append(df)

# Concatenate all DataFrames in the list into a single DataFrame
df_combined_data_dumper = pd.concat(dfs, ignore_index=True)
end_time = time.time()
execution_time = ((end_time - start_time)/60.00)
print("Execution time : ", execution_time, " mins")
df_combined_data_dumper.head(5)

Execution time :  2.7638253132502237  mins


,last_sync_date,user_id,app_list_set
0,20240324,573f28f69b0ffc283676ff35,"['axis mobile', 'microsoft teams', 'whatsapp',..."
1,20240324,573f28f69b0ffc283676ff57,"['whatsapp', 'google photos', 'truecaller', 'a..."
2,20240324,573f28f79b0ffc2836770a38,"['onedrive', 'youtube', 'telegram', 'jio cinem..."
3,20240324,573f28f89b0ffc2836770daf,"['zomato', 'gmail', 'blusmart', 'google maps',..."
4,20240324,573f28f99b0ffc2836771403,"['cleartrip', 'google photos', 'linkedin', 'go..."


#### Optimization & work-around

In [11]:
df_combined_data_dumper.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31723947 entries, 0 to 31723946
Data columns (total 3 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   last_sync_date  int64 
 1   user_id         object
 2   app_list_set    object
dtypes: int64(1), object(2)
memory usage: 726.1+ MB


In [12]:
df_combined_data_dumper['last_sync_date'] = df_combined_data_dumper['last_sync_date'].astype('int32')
df_combined_data_dumper.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31723947 entries, 0 to 31723946
Data columns (total 3 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   last_sync_date  int32 
 1   user_id         object
 2   app_list_set    object
dtypes: int32(1), object(2)
memory usage: 605.1+ MB


In [13]:
## Latest appography sync data
df_combined_data_dumper = df_combined_data_dumper.sort_values(by=['user_id', 'last_sync_date'], ascending=[True, False])
df_combined_data_dumper = df_combined_data_dumper.drop_duplicates(subset='user_id', keep='first')
df_combined_data_dumper.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17829492 entries, 25248151 to 18082310
Data columns (total 3 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   last_sync_date  int32 
 1   user_id         object
 2   app_list_set    object
dtypes: int32(1), object(2)
memory usage: 476.1+ MB


### Merge 

In [14]:
df_city_fe_customer.head(2)

,city,user_id
0,Kochi,5c7cd474875c98536260755b
1,Kochi,65eac9f1e13df5f3f83c6b16


In [15]:
df_combined_data_dumper.rename(columns = {'user_id' : 'customer_id'}, inplace = True)
df_combined_data_dumper.head(2)

,last_sync_date,customer_id,app_list_set
25248151,20240316,5737c6acddbec2203f73315e,"['flipkart', 'disney+ hotstar', 'messenger', '..."
23953873,20240306,5737c6adddbec2203f73316d,"['whatsapp', 'paytm money', 'samsung galaxy we..."


In [16]:
## Merge City Base Customer & Latest appography sync data
df_base_customers_app_data = pd.merge(df_city_fe_customer,
                                      df_combined_data_dumper,
                                      how = 'left',
                                      left_on = ['user_id'],
                                      right_on = ['customer_id']
                                     )
df_base_customers_app_data.head(1)

,city,user_id,last_sync_date,customer_id,app_list_set
0,Kochi,5c7cd474875c98536260755b,20240329.0,5c7cd474875c98536260755b,"['meesho', 'facebook', 'google photos', 'linke..."


In [17]:
df_base_customers_app_data[['city', 'user_id', 'customer_id', 'app_list_set']]\
.to_csv(local_datasets + '{}_city_fe_customer_with_appo_data.csv'.format(city), index=False)

In [18]:
df_base_customers_app_data = pd.read_csv(local_datasets + '{}_city_fe_customer_with_appo_data.csv'.format(city))

In [19]:
df_city_appography_coverage = df_base_customers_app_data\
                                    .groupby(['city'])\
                                    .agg(city_fe_customers = ('user_id','nunique'),
                                         app_info_customers = ('customer_id','nunique')
                                        )\
                                    .reset_index()

df_city_appography_coverage['coverage %'] = (df_city_appography_coverage['app_info_customers']*100/df_city_appography_coverage['city_fe_customers']).round(2)
df_city_appography_coverage

,city,city_fe_customers,app_info_customers,coverage %
0,Kochi,17196,15038,87.45


In [20]:
df_city_appography_coverage

,city,city_fe_customers,app_info_customers,coverage %
0,Kochi,17196,15038,87.45


### Mobile app to category mapping

In [21]:
category_list = pd.read_csv('/Users/rapido/local-datasets/customer/appography/appography_v1/category_list_v1.csv')
category_list['app_name'] = category_list['app_name'].str.lower()
category_list = category_list[['app_name', 'categories_l2', 'categories_l1']]
print(category_list.shape)
print(category_list.categories_l1.unique())
print(category_list.categories_l2.unique())
category_list.head(1)

(399, 3)
['Educational' 'OTT' 'Delivery_Grocery' 'Tools' 'Office' 'Dating' 'News'
 'Devotional' 'Travel_Metro' 'Gaming' 'Travel_Bookings'
 'Finance_Transactions' 'Ecommerce' 'Streaming_Music' 'Finance_Investment'
 'Health' 'Fantasy_Sports' 'Delivery_Food' 'Home_Security' 'Finance_News'
 'Travel_Ridehailing' 'Social' 'Telecom' 'Entertainment' 'News_finance'
 'Wearable' 'Driver_App']
['Educational' 'Rest' 'Office' 'Ride haling' 'Gaming' 'Finance_Investment'
 'Driver_App']


,app_name,categories_l2,categories_l1
0,ignou e-content,Educational,Educational


## Explode mobile app's and mapping categories 

In [22]:
df_customers_with_appography = df_base_customers_app_data[~df_base_customers_app_data['customer_id'].isna()]
df_customers_with_appography.head(4)

,city,user_id,customer_id,app_list_set
0,Kochi,5c7cd474875c98536260755b,5c7cd474875c98536260755b,"['meesho', 'facebook', 'google photos', 'linke..."
1,Kochi,65eac9f1e13df5f3f83c6b16,65eac9f1e13df5f3f83c6b16,"['swiggy', 'samsung galaxy wearable', 'gmail',..."
2,Kochi,62c1bf2717af9a25bb693aa5,62c1bf2717af9a25bb693aa5,"['uber', 'goibibo', 'yatra', 'axis mobile', 'z..."
3,Kochi,65752dd7dd6ceb251cc81a25,65752dd7dd6ceb251cc81a25,"['adobe scan', 'flipkart', 'ola', 'uber', 'you..."


In [23]:
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r"'([^']*)'", text)]

df_customers_with_appography['app_list_set'] = df_customers_with_appography['app_list_set'].apply(lambda x: extract_and_trim(x))
df_customers_with_appography = df_customers_with_appography[['city', 'user_id', 'app_list_set']]
df_customers_with_appography.head(4)

,city,user_id,app_list_set
0,Kochi,5c7cd474875c98536260755b,"[meesho, facebook, google photos, linkedin, gm..."
1,Kochi,65eac9f1e13df5f3f83c6b16,"[swiggy, samsung galaxy wearable, gmail, uber,..."
2,Kochi,62c1bf2717af9a25bb693aa5,"[uber, goibibo, yatra, axis mobile, zomato, ne..."
3,Kochi,65752dd7dd6ceb251cc81a25,"[adobe scan, flipkart, ola, uber, youtube, gro..."


In [24]:
## Explode list into individual rows

df_customers_with_appography_explode = df_customers_with_appography.explode('app_list_set')
df_customers_with_appography_explode['app_list_set'] = df_customers_with_appography_explode['app_list_set'].str.lower()

df_customers_with_appography_explode.head(4)

,city,user_id,app_list_set
0,Kochi,5c7cd474875c98536260755b,meesho
0,Kochi,5c7cd474875c98536260755b,facebook
0,Kochi,5c7cd474875c98536260755b,google photos
0,Kochi,5c7cd474875c98536260755b,linkedin


### Merge

In [25]:
category_list.head(2)

,app_name,categories_l2,categories_l1
0,ignou e-content,Educational,Educational
1,aha,Rest,OTT


In [26]:

df_customers_app_category =  pd.merge(df_customers_with_appography_explode,
                                      category_list, 
                                      how='left', 
                                      left_on='app_list_set',  
                                      right_on='app_name'
                                     )
df_customers_app_category.head(4)

,city,user_id,app_list_set,app_name,categories_l2,categories_l1
0,Kochi,5c7cd474875c98536260755b,meesho,meesho,Rest,Ecommerce
1,Kochi,5c7cd474875c98536260755b,facebook,facebook,Rest,Social
2,Kochi,5c7cd474875c98536260755b,google photos,google photos,Rest,Tools
3,Kochi,5c7cd474875c98536260755b,linkedin,linkedin,Rest,Social


#### Distribution check on categories

In [27]:
total_customers = df_customers_with_appography.user_id.nunique()
total_customers

15038

In [28]:
df_temp = df_customers_app_category\
            .groupby(['categories_l1'])\
            .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                 customers = ('user_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l1,app_list,customers,%
0,Tools,"[google maps, microsoft lens, google photos, t...",15038,100.00
1,Social,"[mx takatak, facebook, instagram, linkedin, sn...",15020,99.88
2,OTT,"[voot, amazon prime video, zee5, sonyliv, alt ...",12143,80.75
3,Ecommerce,"[paytm mall, zivame, firstcry, jiomart, pharme...",12022,79.94
4,Travel_Ridehailing,"[namma yatri, in drive, jugnoo, driveu driver ...",11041,73.42
5,Streaming_Music,"[hungama music, spotify, soundcloud, gaana, am...",10630,70.69
6,Delivery_Food,"[swiggy, dunzo, uber eats, zomato]",10483,69.71
7,Finance_Transactions,"[axis mobile, freecharge, cred, paytm money, m...",10474,69.65
8,Office,"[microsoft teams, pocket, google analytics, gi...",9344,62.14
9,Travel_Bookings,"[indigo, trivago, booking.com, airbnb, yatra, ...",6402,42.57


In [29]:
df_temp = df_customers_app_category\
            .groupby(['categories_l2'])\
            .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                 customers = ('user_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list,customers,%
0,Rest,"[line, google photos, bigbasket, alt balaji, p...",15038,100.00
1,Ride haling,"[namma yatri, in drive, jugnoo, driveu driver ...",11041,73.42
2,Office,"[microsoft teams, pocket, google analytics, gi...",9344,62.14
3,Finance_Investment,"[kite by zerodha, angel broking, hdfc mobile b...",5683,37.79
4,Educational,"[diksha, duolingo, photomath, pocket, vedantu,...",3386,22.52
5,Gaming,[ludo king],1984,13.19
6,Driver_App,"[borzo driver app, rapido captain, swiggy driv...",1281,8.52


In [30]:
df_temp = df_customers_app_category[~df_customers_app_category['categories_l2'].isin(['Rest'])]\
            .groupby(['categories_l2','app_list_set'])\
            .agg(customers = ('user_id','nunique'))\
            .sort_values(by=['categories_l2', 'customers'], ascending=[False, False])\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list_set,customers,%
0,Ride haling,uber,8997,59.83
1,Ride haling,ola,6814,45.31
2,Ride haling,namma yatri,821,5.46
3,Ride haling,quick ride,613,4.08
4,Ride haling,in drive,422,2.81
5,Ride haling,blusmart,134,0.89
6,Ride haling,quickride,50,0.33
7,Ride haling,jugnoo,32,0.21
8,Ride haling,driveu driver app,18,0.12
9,Office,zoom,4318,28.71


## Transform Categories 

In [31]:
df_customers_app_category['categories_l1'] = df_customers_app_category['categories_l1'].str.lower()
df_customers_app_category['categories_l2'] = df_customers_app_category['categories_l2'].str.lower()
df_customers_app_category.head(4)

,city,user_id,app_list_set,app_name,categories_l2,categories_l1
0,Kochi,5c7cd474875c98536260755b,meesho,meesho,rest,ecommerce
1,Kochi,5c7cd474875c98536260755b,facebook,facebook,rest,social
2,Kochi,5c7cd474875c98536260755b,google photos,google photos,rest,tools
3,Kochi,5c7cd474875c98536260755b,linkedin,linkedin,rest,social


In [32]:
df_customer_mapped = df_customers_app_category\
                        .groupby(['city', 'user_id'])\
                        .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                             categories_l1 = ('categories_l1', lambda x: list(set(x))),
                             categories_l2 = ('categories_l2', lambda x: list(set(x)))
                            )\
                        .reset_index()

df_customer_mapped.head(4)

,city,user_id,app_list,categories_l1,categories_l2
0,Kochi,573f28f39b0ffc283676f2d9,"[facebook, spotify, bookmyshow, google maps, g...","[office, tools, social, entertainment, wearabl...","[office, ride haling, rest]"
1,Kochi,573f28f89b0ffc2836770f5f,"[microsoft teams, amazon prime video, facebook...","[ott, office, tools, delivery_grocery, social,...","[office, ride haling, rest, finance_investment]"
2,Kochi,573f29039b0ffc28367736a7,"[microsoft teams, amazon prime video, facebook...","[ott, office, tools, social, entertainment, ne...","[office, ride haling, rest, finance_investment]"
3,Kochi,573f29099b0ffc283677425c,"[amazon prime video, facebook, bookmyshow, goo...","[ott, office, tools, delivery_grocery, social,...","[office, ride haling, rest]"


In [33]:
unique_categories = set(category for sublist in df_customer_mapped['categories_l1'] for category in sublist)
unique_categories

{'dating',
 'delivery_food',
 'delivery_grocery',
 'driver_app',
 'ecommerce',
 'educational',
 'entertainment',
 'fantasy_sports',
 'finance_investment',
 'finance_news',
 'finance_transactions',
 'gaming',
 'health',
 'news',
 'office',
 'ott',
 'social',
 'streaming_music',
 'tools',
 'travel_bookings',
 'travel_ridehailing',
 'wearable'}

In [34]:
print("Count of unique categories:", len(unique_categories))

Count of unique categories: 22


In [35]:
for category in unique_categories:
    df_customer_mapped[category] = df_customer_mapped['categories_l1'].apply(lambda x: 1 if category in x else 0)
df_customer_mapped.head(1)

,city,user_id,app_list,categories_l1,categories_l2,ott,office,social,entertainment,news,gaming,travel_ridehailing,delivery_food,streaming_music,ecommerce,delivery_grocery,health,finance_news,finance_investment,driver_app,educational,fantasy_sports,tools,wearable,dating,travel_bookings,finance_transactions
0,Kochi,573f28f39b0ffc283676f2d9,"[facebook, spotify, bookmyshow, google maps, g...","[office, tools, social, entertainment, wearabl...","[office, ride haling, rest]",0,1,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,1,1,0,1,0


In [40]:
# df_customer_mapped['devotional'] = 0

In [41]:
df_customer_mapped.columns

Index(['city', 'user_id', 'app_list', 'categories_l1', 'categories_l2', 'ott',
       'office', 'social', 'entertainment', 'news', 'gaming',
       'travel_ridehailing', 'delivery_food', 'streaming_music', 'ecommerce',
       'delivery_grocery', 'health', 'finance_news', 'finance_investment',
       'driver_app', 'educational', 'fantasy_sports', 'tools', 'wearable',
       'dating', 'travel_bookings', 'finance_transactions', 'devotional'],
      dtype='object')

In [42]:
df_city_final_dataset = df_customer_mapped[['city', 'user_id',
                                           'dating', 'delivery_food', 'delivery_grocery', 'devotional', 
                                            'driver_app', 'ecommerce', 'educational', 'entertainment', 
                                            'fantasy_sports', 'finance_investment', 'finance_news', 
                                            'finance_transactions', 'gaming', 'health', 'news', 
                                            'office', 'ott', 'social', 'streaming_music', 'tools', 
                                            'travel_bookings', 'travel_ridehailing', 'wearable']]

In [43]:
df_city_final_dataset.head(5)

,city,user_id,dating,delivery_food,delivery_grocery,devotional,driver_app,ecommerce,educational,entertainment,fantasy_sports,finance_investment,finance_news,finance_transactions,gaming,health,news,office,ott,social,streaming_music,tools,travel_bookings,travel_ridehailing,wearable
0,Kochi,573f28f39b0ffc283676f2d9,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,1,1,1,1
1,Kochi,573f28f89b0ffc2836770f5f,0,1,1,0,0,1,0,1,0,1,0,1,0,0,1,1,1,1,0,1,1,1,0
2,Kochi,573f29039b0ffc28367736a7,0,1,0,0,0,1,0,1,0,1,0,1,0,0,1,1,1,1,1,1,1,1,0
3,Kochi,573f29099b0ffc283677425c,0,1,1,0,0,1,0,1,0,0,0,1,0,0,0,1,1,1,0,1,0,1,1
4,Kochi,573f290d9b0ffc2836775d6a,0,1,0,0,0,1,0,1,0,0,0,1,0,0,0,1,1,1,1,1,1,1,0


In [44]:
df_city_final_dataset.rename(columns = {'user_id' : 'customer_id'}, inplace=True)

In [45]:
df_city_final_dataset[df_city_final_dataset['customer_id'] == '573f28f39b0ffc283676f2d9']

,city,customer_id,dating,delivery_food,delivery_grocery,devotional,driver_app,ecommerce,educational,entertainment,fantasy_sports,finance_investment,finance_news,finance_transactions,gaming,health,news,office,ott,social,streaming_music,tools,travel_bookings,travel_ridehailing,wearable
0,Kochi,573f28f39b0ffc283676f2d9,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,1,1,1,1


## Final dataset

In [46]:
df_city_final_dataset

,city,customer_id,dating,delivery_food,delivery_grocery,devotional,driver_app,ecommerce,educational,entertainment,fantasy_sports,finance_investment,finance_news,finance_transactions,gaming,health,news,office,ott,social,streaming_music,tools,travel_bookings,travel_ridehailing,wearable
0,Kochi,573f28f39b0ffc283676f2d9,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,1,1,1,1
1,Kochi,573f28f89b0ffc2836770f5f,0,1,1,0,0,1,0,1,0,1,0,1,0,0,1,1,1,1,0,1,1,1,0
2,Kochi,573f29039b0ffc28367736a7,0,1,0,0,0,1,0,1,0,1,0,1,0,0,1,1,1,1,1,1,1,1,0
3,Kochi,573f29099b0ffc283677425c,0,1,1,0,0,1,0,1,0,0,0,1,0,0,0,1,1,1,0,1,0,1,1
4,Kochi,573f290d9b0ffc2836775d6a,0,1,0,0,0,1,0,1,0,0,0,1,0,0,0,1,1,1,1,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15033,Kochi,660990fd23569dbc3a239fa3,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,1,1,1,0,0,0
15034,Kochi,6609931adb0e9c4cff3c1d01,0,0,0,0,0,1,0,1,1,1,0,1,0,0,0,1,1,1,1,1,0,1,0
15035,Kochi,66099b42efc6d9826cf2ddf3,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,1,0,1,0,1,1,1,0
15036,Kochi,66099b675a39711d0ce0803b,1,1,0,0,0,1,0,1,0,0,0,1,0,0,1,0,1,1,1,1,1,1,0


In [47]:
df_city_final_dataset.to_csv(local_datasets + '{}_city_appography_dataset.csv'.format(city), index=False)

In [48]:
df_city_final_dataset_test = pd.read_csv(local_datasets + '{}_city_appography_dataset.csv'.format(city))

In [49]:
df_city_final_dataset_test

,city,customer_id,dating,delivery_food,delivery_grocery,devotional,driver_app,ecommerce,educational,entertainment,fantasy_sports,finance_investment,finance_news,finance_transactions,gaming,health,news,office,ott,social,streaming_music,tools,travel_bookings,travel_ridehailing,wearable
0,Kochi,573f28f39b0ffc283676f2d9,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,1,1,1,1
1,Kochi,573f28f89b0ffc2836770f5f,0,1,1,0,0,1,0,1,0,1,0,1,0,0,1,1,1,1,0,1,1,1,0
2,Kochi,573f29039b0ffc28367736a7,0,1,0,0,0,1,0,1,0,1,0,1,0,0,1,1,1,1,1,1,1,1,0
3,Kochi,573f29099b0ffc283677425c,0,1,1,0,0,1,0,1,0,0,0,1,0,0,0,1,1,1,0,1,0,1,1
4,Kochi,573f290d9b0ffc2836775d6a,0,1,0,0,0,1,0,1,0,0,0,1,0,0,0,1,1,1,1,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15033,Kochi,660990fd23569dbc3a239fa3,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,1,1,1,0,0,0
15034,Kochi,6609931adb0e9c4cff3c1d01,0,0,0,0,0,1,0,1,1,1,0,1,0,0,0,1,1,1,1,1,0,1,0
15035,Kochi,66099b42efc6d9826cf2ddf3,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,1,0,1,0,1,1,1,0
15036,Kochi,66099b675a39711d0ce0803b,1,1,0,0,0,1,0,1,0,0,0,1,0,0,1,0,1,1,1,1,1,1,0


In [50]:
df_city_final_dataset_test.head(5).T

,0,1,2,3,4
city,Kochi,Kochi,Kochi,Kochi,Kochi
customer_id,573f28f39b0ffc283676f2d9,573f28f89b0ffc2836770f5f,573f29039b0ffc28367736a7,573f29099b0ffc283677425c,573f290d9b0ffc2836775d6a
dating,0,0,0,0,0
delivery_food,1,1,1,1,1
delivery_grocery,0,1,0,1,0
devotional,0,0,0,0,0
driver_app,0,0,0,0,0
ecommerce,0,1,1,1,1
educational,0,0,0,0,0
entertainment,1,1,1,1,1


In [51]:
df_city_final_dataset_test.to_csv(local_datasets + 'v0_{}_city_appography_dataset.csv'.format(city), 
                                  header=False, 
                                  index=False)